# 군집분석
- 유사성이 높은 데이터를 그룹화하여(군집화) 데이터의 내재된 구조를 파악하는 비지도 학습(Unsupervised Learning) 방법입니다. 생물학쪽에서는 Hierarchical Clustering을 많이 하는데... 어... 이거 잘하면 계통수 대체 가능하겠는데?
- 여기서는 쇼핑 데이터를 이용해서 군집분석을 해보겠습니다. 그래서 이건 EDA가 아니고 통계실전시리즈임.

In [ ]:
from unicodedata import category

import kagglehub

# Download latest version
path = kagglehub.dataset_download("wardabilal/customer-shopping-behaviour-analysis")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 군집분석용
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.cluster.hierarchy import fcluster
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler # PCA에서 많이 보인 그 분
from sklearn.metrics import pairwise_distances # 저친구가 거리행렬을 안주면 유혈사태(아니고 에러사태)가 납니다

# k-means
from sklearn.cluster import KMeans

# k-medoid (이거 따로 까서야돼요)
import kmedoids

# 누구세요?
from sklearn.manifold import TSNE

# 통계분석용
from scipy import stats

In [ ]:
# 그래프 기본 테마 설정
sns.set_theme(palette="Set2", style="whitegrid", font_scale=1)  # 블루톤

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Mabinogi_Classic'  # 제가... 픽셀체 이런거 좋아해서...
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['axes.titlesize'] = 16  # 제목 폰트 사이즈
plt.rcParams['axes.labelsize'] = 14  # 라벨 폰트 사이즈
plt.rcParams['font.size'] = 14  # 기본 폰트사이즈
plt.rcParams['axes.unicode_minus'] = False

# 데이터 불러오기 및 정보 확인

In [ ]:
shopping_df = pd.read_csv(f'{path}/shopping_behavior_updated (1).csv')
shopping_df

## .info()

In [ ]:
shopping_df.info()

## .describe()

In [ ]:
shopping_df.describe()

In [ ]:
shopping_df.describe(include='O')

## .isna().sum()

In [ ]:
shopping_df.isna().sum()

## .columns

In [ ]:
shopping_df.columns

## .head()

In [ ]:
shopping_df.head()

# 군집분석
## 계층적 군집분석(Hierarchical clustering)

### 나이

In [ ]:
data = shopping_df[['Age']] # 나이로
Z = linkage(data, method='ward') # 묶어보시오

# 덴드로그램
plt.figure(figsize=(16, 9))
dendrogram(Z, truncate_mode='lastp',p=12,leaf_rotation=45,leaf_font_size=12,show_contracted=True)
plt.show()

- 뭔진 몰라도 일단 망한건 확실합니다.

 ### 구매품목

In [ ]:
# 혹시 묶을게 범주형입니까? 원 핫 인코딩 ㄱㄱ하세요.
category = np.array(shopping_df['Item Purchased']) # 구매한 아이템
category = category.reshape(-1, 1)

encoder = OneHotEncoder().fit(category) ## 범주와 One-Hot Encoding간 매핑 생성
sparse_mat = encoder.transform(category) ## 실제로 변환할 때에는 transform 사용
sparse_mat = sparse_mat.toarray()

Z = linkage(sparse_mat, method='ward') # 묶어보시오

# 덴드로그램
plt.figure(figsize=(16, 9))
dendrogram(Z, truncate_mode='lastp',p=12,leaf_rotation=45,leaf_font_size=12,show_contracted=True)
plt.show()

- 아니 요약한 게 이거라고?? 이거 원래 이래요?

### 나이+결제수단

In [ ]:
data = shopping_df[['Age']] # 나이
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data) # 를 스케일링합니다

In [ ]:
category = np.array(shopping_df['Payment Method']) # 구매 방법
category = category.reshape(-1, 1)

encoder = OneHotEncoder().fit(category) ## 범주와 One-Hot Encoding간 매핑 생성
sparse_mat = encoder.transform(category) ## 실제로 변환할 때에는 transform 사용
sparse_mat = sparse_mat.toarray()

In [ ]:
combined_data = np.hstack([data_scaled, sparse_mat]) # 파이널- 퓨-전!!!

- 자 봐봐요. 나이는 스탠다드 스케일러로 0~1까지로 만들었죠? 그리고 결제수단은 범주형이니까 원 핫 인코딩을 했어요.
- 그럼 두개가 나오는데 두개를 그냥 넣겠음? 쟤네가 그렇게 설계하질 않았어요. 그럼 어떻게 하냐고요? 합쳐야죠.

In [ ]:
Z = linkage(combined_data, method='ward') # 묶어보시오

# 덴드로그램
plt.figure(figsize=(16, 9))
dendrogram(Z, truncate_mode='lastp',p=12,leaf_rotation=45,leaf_font_size=12,show_contracted=True)
plt.show()

- 뭔가 나왔습니다. 근데... 우리 솔직히 저 가지에 뭐 있는지 궁금하지 않아요?

In [ ]:
k = 4 # 원하는 군집 개수 (그래프를 보고 4개 정도로 정해봅시다)
clusters = fcluster(Z, k, criterion='maxclust') # 군집이 요기잉눼?

# 원본 데이터에서 확인할 수 있게 해줍니다.
shopping_df['Cluster'] = clusters

# 군집별 특징 확인 (평균 나이 등)
cluster_summary = shopping_df.groupby('Cluster').agg({
    'Age': 'mean',
    'Payment Method': lambda x: x.mode()[0] # 각 군집이 가장 많이 사용한 결제수단
}).reset_index()

print(cluster_summary)

- Venmo는 그 약간... 카카오페이 뭐 그런거래요.

In [ ]:
plt.figure(figsize=(10, 6))
# 군집별로 나이 분포가 얼마나 다른지 확인
sns.boxplot(x='Cluster', y='Age', data=shopping_df)
plt.title('Cluster별 나이 분포 확인 (Venmo는 몇 살일까?)')
plt.show()

- 좀 의외인데요? 보통은 어르신들이 현금을 많이 쓰고(아니면 카드) 젊은 층들이 페이류를 좀 많이 쓰지 않나...?

In [ ]:
# 구매 금액(Purchase Amount (USD))까지 포함해서 요약
cluster_final_check = shopping_df.groupby('Cluster').agg({
    'Age': 'mean',
    'Payment Method': lambda x: x.mode()[0],
    'Purchase Amount (USD)': 'mean'  # 평균 구매 금액 추가!
}).reset_index()

print(cluster_final_check)

#### ANOVA
- 솔직히 저거 진짜로 유의한 차이인지 궁금하지 않아요?

In [ ]:
# 1. 군집별로 구매 금액 데이터 분리
g1 = shopping_df[shopping_df['Cluster'] == 1]['Purchase Amount (USD)']
g2 = shopping_df[shopping_df['Cluster'] == 2]['Purchase Amount (USD)']
g3 = shopping_df[shopping_df['Cluster'] == 3]['Purchase Amount (USD)']
g4 = shopping_df[shopping_df['Cluster'] == 4]['Purchase Amount (USD)']

# 2. 일원배치 분산분석(One-way ANOVA) 수행
f_stat, p_val = stats.f_oneway(g1, g2, g3, g4)

print(f'F-통계량: {f_stat:.4f}')
print(f'p-value: {p_val:.4f}')

# 3. 결과 해석
if p_val < 0.05:
    print("결과: 군집 간 평균 구매 금액에 유의미한 차이가 있습니다! (귀무가설 기각)")
else:
    print("결과: 군집 간 평균 구매 금액 차이는 통계적으로 유의미하지 않습니다. (귀무가설 채택)")

### 나이+뭐샀음?

In [ ]:
# 나이는 위에 있는거 걍 쓸게요
category = np.array(shopping_df['Item Purchased']) # 구매 방법
category = category.reshape(-1, 1)

encoder = OneHotEncoder().fit(category) ## 범주와 One-Hot Encoding간 매핑 생성
sparse_mat = encoder.transform(category) ## 실제로 변환할 때에는 transform 사용
sparse_mat = sparse_mat.toarray()

In [ ]:
combined_data = np.hstack([data_scaled, sparse_mat]) # 파이널- 퓨-전!!!

In [ ]:
Z = linkage(combined_data, method='ward') # 묶어보시오

# 덴드로그램
plt.figure(figsize=(16, 9))
dendrogram(Z, truncate_mode='lastp',p=12,leaf_rotation=45,leaf_font_size=12,show_contracted=True)
plt.show()

- 아까랑은 양상이 좀 다르죠?

In [ ]:
k = 6 # 원하는 군집 개수 (그래프를 보고 4개 정도로 정해봅시다)
clusters = fcluster(Z, k, criterion='maxclust') # 군집이 요기잉눼?

# 원본 데이터에서 확인할 수 있게 해줍니다.
shopping_df['Cluster'] = clusters

# 군집별 특징 확인 (평균 나이 등)
cluster_summary = shopping_df.groupby('Cluster').agg({
    'Age': 'mean',
    'Item Purchased': lambda x: x.mode()[0] # 각 군집이 가장 많이 사용한 결제수단
}).reset_index()

print(cluster_summary)

In [ ]:
# 군집별 아이템 구매 빈도 확인
ct = pd.crosstab(shopping_df['Cluster'], shopping_df['Item Purchased'])

plt.figure(figsize=(12, 8))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu')
plt.title('군집별 구매 품목 집중도')
plt.show()

In [ ]:
# 1번 군집만 필터링해서 아이템 순위 보기
cluster_1 = shopping_df[shopping_df['Cluster'] == 1]
print("1번 군집(60대)의 쇼핑백 TOP 5:")
print(cluster_1['Item Purchased'].value_counts().head(5))

In [ ]:
# 1번 군집의 선호 색상 확인
print("1번 군집의 상견례룩(?) 컬러 TOP 3:")
print(cluster_1['Color'].value_counts().head(3))

- 쥬얼리라길래 금혼식인 줄 알았더니 상견례 최강자 룩 찾고 계셨던거냐고...

In [ ]:
# 구매 금액(Purchase Amount (USD))까지 포함해서 요약
cluster_final_check = shopping_df.groupby('Cluster').agg({
    'Age': 'mean',
    'Payment Method': lambda x: x.mode()[0],
    'Purchase Amount (USD)': 'mean'  # 평균 구매 금액 추가!
}).reset_index()

print(cluster_final_check)

## K-mean
- 주어진 데이터를 k개의 클러스터로 묶는 알고리즘 ~~K가 Korea의 K가 아닙니다~~

In [ ]:
# 1. Inertia(오차 제곱합) 값을 저장할 리스트
inertia = []
k_range = range(1, 11)  # 1개부터 10개까지 군집 개수를 늘려가며 확인

# 2. 반복문으로 각 K에 대한 모델 학습
for k in k_range:
    # n_init=10: 초기 중심점을 10번 다르게 잡아서 최적의 결과를 선택
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(combined_data) # 나이(Scaled) + 결제수단(One-Hot) 데이터
    inertia.append(kmeans.inertia_)

# 3. 그래프 시각화 (설정하신 NanumSquare가 적용됩니다)
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o', color='forestgreen', linewidth=2)
plt.title('Optimal K 찾기 (Elbow Method)', fontsize=15)
plt.xlabel('군집 개수 (K)', fontsize=12)
plt.ylabel('Inertia (오차 제곱합)', fontsize=12)
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.show()

- 엘보가 2에서 끝나지요.

In [ ]:
# 1. K=2 버전 (수학적 최적)
kmeans2 = KMeans(n_clusters=2, random_state=42, n_init=10)
shopping_df['KMeans_K2'] = kmeans2.fit_predict(combined_data)

# 2. K=4 버전 (분석가 추천)
kmeans4 = KMeans(n_clusters=4, random_state=42, n_init=10)
shopping_df['KMeans_K4'] = kmeans4.fit_predict(combined_data)

# 3. 결과 비교를 위한 요약표 출력
print("--- K=2 결과 요약 ---")
print(shopping_df.groupby('KMeans_K2')['Age'].agg(['mean', 'count']))
print("\n--- K=4 결과 요약 ---")
print(shopping_df.groupby('KMeans_K4')['Age'].agg(['mean', 'count']))

### 시각화

In [ ]:
# 1. PCA 대신 t-SNE로 교체
# 아이 이 에미나이 왜이렇게 PCA를 좋아해!
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_data = tsne.fit_transform(combined_data)
shopping_df['pca_x'] = tsne_data[:, 0]
shopping_df['pca_y'] = tsne_data[:, 1]

# 2. 그래프 그리기 (NanumSquare 폰트 적용)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# K=2 시각화
sns.scatterplot(data=shopping_df, x='pca_x', y='pca_y', hue='KMeans_K2', palette='viridis', ax=ax1)
ax1.set_title('K-Means 군집 시각화 (K=2)', fontsize=14)

# K=4 시각화
sns.scatterplot(data=shopping_df, x='pca_x', y='pca_y', hue='KMeans_K4', palette='Set2', ax=ax2)
ax2.set_title('K-Means 군집 시각화 (K=4)', fontsize=14)

plt.tight_layout()
plt.show()

- 너무 이븐한거 아니냐고... ~~그래프가 이븐하게 펼쳐져 있어요. 그래프가 고루 펼쳐져 있어요. 생존입니다.~~

## K-medoid
- 위에 친구는 평균으로 하고요, 이 친구는 중앙값으로 합니다.

In [ ]:
# 거리 행렬 & K-Medoids
dist_matrix = pairwise_distances(combined_data, metric='manhattan')
km_model = kmedoids.KMedoids(n_clusters=2, method='fasterpam', random_state=42)  # K=2로!
km_result = km_model.fit(dist_matrix)

medoid_indices = km_result.medoid_indices_
representative_customers = shopping_df.iloc[medoid_indices]
print(representative_customers[['Age', 'Item Purchased', 'Payment Method']])

### 시각화

In [ ]:
# 1. 시각화를 위해 PCA 데이터에 K-medoids 결과 합치기
shopping_df['KMedoids_Cluster'] = km_result.labels_ # 아까 돌린 kmedoids 결과

# 2. 산점도 그리기
plt.figure(figsize=(12, 8))

# 전체 데이터 뿌리기
sns.scatterplot(
    data=shopping_df,
    x='pca_x', y='pca_y',
    hue='KMedoids_Cluster',
    palette='deep',
    alpha=0.6,
    edgecolor=None
)

# 3. '진짜 대표님(Medoids)'들 위치에 별표 찍기
medoid_pca = shopping_df.iloc[medoid_indices]
plt.scatter(
    medoid_pca['pca_x'],
    medoid_pca['pca_y'],
    marker='*',
    s=450,        # 별 크기
    c='red',      # 강렬한 레드
    label='Medoids (대표님)',
    edgecolor='black'
)

plt.title('K-medoids 군집 시각화 및 대표 데이터(★) 위치', fontsize=16)
plt.xlabel('주성분 1 (나이 중심 축)', fontsize=12)
plt.ylabel('주성분 2 (기타 변수 축)', fontsize=12)
plt.legend(loc=1, bbox_to_anchor=(1.2, 1))
plt.grid(True, alpha=0.3)
plt.show()

## 나이+별점으로 재도전
- 아니 왕꿈틀이가 왜 나와요 ㅋㅋㅋㅋㅋㅋ

In [ ]:
# 나이 + 별점으로 재구성
data = shopping_df[['Age', 'Purchase Amount (USD)']]  # 별점 컬럼명이 뭐예요?
scaler = StandardScaler()
combined_data = scaler.fit_transform(data)

### 팔꿈치 찾기

In [ ]:
# 1. Inertia(오차 제곱합) 값을 저장할 리스트
inertia = []
k_range = range(1, 11)  # 1개부터 10개까지 군집 개수를 늘려가며 확인

# 2. 반복문으로 각 K에 대한 모델 학습
for k in k_range:
    # n_init=10: 초기 중심점을 10번 다르게 잡아서 최적의 결과를 선택
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(combined_data) # 나이(Scaled) + 결제수단(One-Hot) 데이터
    inertia.append(kmeans.inertia_)

# 3. 그래프 시각화 (설정하신 NanumSquare가 적용됩니다)
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o', color='forestgreen', linewidth=2)
plt.title('Optimal K 찾기 (Elbow Method)', fontsize=15)
plt.xlabel('군집 개수 (K)', fontsize=12)
plt.ylabel('Inertia (오차 제곱합)', fontsize=12)
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.show()

### K-means & K-medoids

In [ ]:
# K-Means
kmeans4 = KMeans(n_clusters=4, random_state=42, n_init=10)
shopping_df['KMeans_K4'] = kmeans4.fit_predict(combined_data)

# 거리 행렬 & K-Medoids
dist_matrix = pairwise_distances(combined_data, metric='euclidean')
km_model = kmedoids.KMedoids(n_clusters=4, method='fasterpam', random_state=42)
km_result = km_model.fit(dist_matrix)
shopping_df['KMedoids_K4'] = km_result.labels_

# 대표 고객
medoid_indices = km_result.medoid_indices_
representative_customers = shopping_df.iloc[medoid_indices]
print(representative_customers[['Age', 'Purchase Amount (USD)']])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

sns.scatterplot(data=shopping_df, x='Age', y='Purchase Amount (USD)',
                hue='KMeans_K4', palette='Set2', ax=ax)
ax.set_title('K-Means 군집 시각화 (K=4)')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

sns.scatterplot(data=shopping_df, x='Age', y='Purchase Amount (USD)',
                hue='KMedoids_K4', palette='Set2', ax=ax)
ax.set_title('K-Means 군집 시각화 (K=4)')

plt.tight_layout()
plt.show()